# Fatigue analysis


In [1]:
from fitparse import FitFile

fit = FitFile('/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/EcoTrail de Paris_2026-03-21 10:55:00.fit')

# Chercher tous les types de messages disponibles
msg_types = set()
for msg in fit.get_messages():
    msg_types.add(msg.name)
print("Types de messages :", sorted(msg_types))

# Regarder les 10 premiers records avec TOUS leurs champs
fit2 = FitFile('/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/EcoTrail de Paris_2026-03-21 10:55:00.fit')
for i, rec in enumerate(fit2.get_messages('record')):
    d = {f.name: f.value for f in rec}
    print(f"Record {i}: {d}")
    if i >= 9:
        break

Types de messages : ['activity', 'developer_data_id', 'device_info', 'event', 'field_description', 'file_id', 'hrv', 'lap', 'record', 'session']
Record 0: {'distance': 0.0, 'timestamp': datetime.datetime(2026, 3, 21, 9, 55, 41)}
Record 1: {'position_lat': 582083934, 'position_long': 23434078, 'timestamp': datetime.datetime(2026, 3, 21, 9, 55, 42)}
Record 2: {'heart_rate': 109, 'position_lat': 582084212, 'position_long': 23434514, 'timestamp': datetime.datetime(2026, 3, 21, 9, 55, 44)}
Record 3: {'heart_rate': 111, 'position_lat': 582084391, 'position_long': 23434833, 'timestamp': datetime.datetime(2026, 3, 21, 9, 55, 45)}
Record 4: {'heart_rate': 112, 'position_lat': 582084650, 'position_long': 23435191, 'timestamp': datetime.datetime(2026, 3, 21, 9, 55, 46)}
Record 5: {'heart_rate': 113, 'position_lat': 582084868, 'position_long': 23435528, 'timestamp': datetime.datetime(2026, 3, 21, 9, 55, 47)}
Record 6: {'heart_rate': 115, 'position_lat': 582085086, 'position_long': 23435847, 'times

In [2]:
import sys, os
sys.path.insert(0, '../core')

from fitparse import FitFile
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from gpx_race import haversine
from race_predictor import minetti_speed_ratio

DATA_FIT = '/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton'

RACES_FIT = {
    'EcoTrail Paris 2026':       'EcoTrail de Paris_2026-03-21 10:55:00.fit',
    'Imperial Fontainebleau 2025': 'Imperial trail de Fontainebleau_2025-09-13 06:01:00.fit',
    'SainteLyon 2025':           'SainteLyon_2025-11-29 23:31:00.fit',
}

SMOOTH_WIN_S = 300   # fenêtre de lissage vitesse (secondes)
N_BINS = 20          # nombre de bins par % de distance


def load_fit_records(path):
    """
    Load FIT file and return list of record dicts with all fields merged by timestamp.
    
    FIT records are often fragmented across multiple messages sharing the same
    timestamp. This function merges them into a single dict per timestamp.
    """
    from collections import defaultdict
    fit = FitFile(path)

    by_ts = defaultdict(dict)
    for rec in fit.get_messages('record'):
        d = {f.name: f.value for f in rec}
        ts = d.get('timestamp')
        if ts is not None:
            by_ts[ts].update(d)

    records = [v for _, v in sorted(by_ts.items(), key=lambda x: x[0])]
    records = [r for r in records if r.get('distance') is not None]
    return records


def compute_smoothed_speed(records, smooth_win_s):
    """Compute smoothed speed (km/h) and distance (km) arrays."""
    n = len(records)
    speeds, dists = [], []
    for i in range(n):
        t0 = records[i]['timestamp']
        d0 = records[i]['distance']
        j = i
        while j < n - 1 and (records[j]['timestamp'] - t0).total_seconds() < smooth_win_s:
            j += 1
        dt = (records[j]['timestamp'] - t0).total_seconds()
        dd = records[j]['distance'] - d0
        if dt > 10 and dd >= 0:
            speeds.append(dd / dt * 3.6)
            dists.append(d0 / 1000)
    return np.array(dists), np.array(speeds)


# def compute_slope_from_records(records):
#     """Estimate slope (%) at each record point from GPS coordinates."""
#     lats = [r.get('position_lat') for r in records]
#     lons = [r.get('position_long') for r in records]
#     dists = [r.get('distance', 0) for r in records]

#     # Coordonnées en degrés (FIT stocke en semicircles)
#     lats = [l * 180 / 2**31 if l else None for l in lats]
#     lons = [l * 180 / 2**31 if l else None for l in lons]

#     # Pas d'altitude dans ces fichiers — pente estimée à 0
#     # On retourne un tableau de zéros (correction Minetti neutre)
#     return np.zeros(len(records))


# def fatigue_profile(records, smooth_win_s, n_bins):
#     """
#     Compute ratio v_real / v_minetti per distance bin.

#     Returns bin centers (pct distance) and median ratios.
#     """
#     dists, speeds = compute_smoothed_speed(records, smooth_win_s)
#     total = dists.max()

#     # Pente estimée à chaque point (0 ici, fichiers sans altitude)
#     slopes = np.zeros(len(dists))
#     v_minetti_ratio = np.array([minetti_speed_ratio(s) for s in slopes])

#     # Vitesse de référence sur terrain plat : médiane des 20 premiers %
#     mask_ref = dists < 0.20 * total
#     v_ref = np.median(speeds[mask_ref & (speeds > 1.5)]) if mask_ref.sum() > 10 else np.nan

#     # Ratio réel / attendu Minetti
#     v_expected = v_ref * v_minetti_ratio
#     ratio = np.where(v_expected > 0.5, speeds / v_expected, np.nan)

#     bins = np.linspace(0, total, n_bins + 1)
#     bin_centers, bin_ratios = [], []
#     for i in range(n_bins):
#         mask = (dists >= bins[i]) & (dists < bins[i+1]) & (speeds > 1.5)
#         med = np.nanmedian(ratio[mask]) if mask.sum() > 5 else np.nan
#         bin_centers.append((i + 0.5) / n_bins)
#         bin_ratios.append(med)

#     return np.array(bin_centers), np.array(bin_ratios), v_ref, total

def compute_slope_from_records(records, smooth_win_m=200):
    """
    Estimate slope (%) at each record point from enhanced_altitude and distance.
    Smoothed over smooth_win_m meters.
    """
    n     = len(records)
    dists = np.array([r.get('distance', 0) for r in records])
    alts  = np.array([r.get('enhanced_altitude') or 0 for r in records], dtype=float)

    # Lissage de l'altitude avant calcul de pente
    win = min(21, n if n % 2 == 1 else n - 1)
    win = max(win, 5)
    if n >= win:
        alts = savgol_filter(alts, window_length=win, polyorder=3)

    slopes = np.zeros(n)
    for i in range(n):
        d0   = dists[i]
        mask = np.abs(dists - d0) < smooth_win_m / 2
        if mask.sum() < 3:
            continue
        d_range = dists[mask]
        a_range = alts[mask]
        dd = d_range[-1] - d_range[0]
        da = a_range[-1] - a_range[0]
        slopes[i] = da / dd * 100 if dd > 0 else 0.0

    return np.clip(slopes, -45, 45)


def fatigue_profile(records, smooth_win_s, n_bins):
    dists, speeds = compute_smoothed_speed(records, smooth_win_s)
    print(f"  speeds shape: {speeds.shape}, dists shape: {dists.shape}")
    
    total = dists.max()
    print(f"  total: {total}")

    slopes_all = compute_slope_from_records(records, smooth_win_m=200)
    print(f"  slopes_all shape: {slopes_all.shape}")
    
    dists_all = np.array([r.get('distance', 0) / 1000 for r in records])
    slopes_interp = np.interp(dists, dists_all, slopes_all)
    print(f"  slopes_interp shape: {slopes_interp.shape}")

    v_minetti_ratio = np.array([minetti_speed_ratio(s) for s in slopes_interp])
    print(f"  v_minetti_ratio shape: {v_minetti_ratio.shape}")

    mask_ref = (dists < 0.20 * total) & (speeds > 1.5) & (np.abs(slopes_interp) < 3)
    print(f"  mask_ref sum: {mask_ref.sum()}")
    if mask_ref.sum() > 10:
        v_ref = np.median(speeds[mask_ref])
    else:
        v_ref = np.median(speeds[speeds > 1.5][:100])
    print(f"  v_ref: {v_ref}")

    v_expected = v_ref * v_minetti_ratio
    ratio = np.where(v_expected > 0.5, speeds / v_expected, np.nan)

    bins = np.linspace(0, total, n_bins + 1)
    bin_centers, bin_ratios = [], []
    for i in range(n_bins):
        mask = (dists >= bins[i]) & (dists < bins[i+1]) & (speeds > 1.5)
        med  = np.nanmedian(ratio[mask]) if mask.sum() > 5 else np.nan
        bin_centers.append((i + 0.5) / n_bins)
        bin_ratios.append(med)

    return np.array(bin_centers), np.array(bin_ratios), v_ref, total

# Calcul sur toutes les courses
profiles = {}
for name, fname in RACES_FIT.items():
    pathfile = os.path.join(DATA_FIT, fname)
    print(pathfile)


    records = load_fit_records(pathfile)
    print(f"  Nb records : {len(records)}")
    print(f"  Champs dispo : {set(records[0].keys())}")
    print(f"  enhanced_altitude sample : {[r.get('enhanced_altitude') for r in records[:5]]}")



    centers, ratios, v_ref, total_km = fatigue_profile(records, SMOOTH_WIN_S, N_BINS)
    profiles[name] = {
        'centers': centers,
        'ratios':  ratios,
        'v_ref':   v_ref,
        'total_km': total_km,
    }
    print(f"{name} ({total_km:.1f} km) — v_ref={v_ref:.2f} km/h")

/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/EcoTrail de Paris_2026-03-21 10:55:00.fit
  Nb records : 38005
  Champs dispo : {'distance', 'timestamp'}
  enhanced_altitude sample : [None, 168.79999999999995, 169.0, 169.0, 169.0]
  speeds shape: (37994,), dists shape: (37994,)
  total: 81.393
  slopes_all shape: (38005,)
  slopes_interp shape: (37994,)
  v_minetti_ratio shape: (37994,)
  mask_ref sum: 4902
  v_ref: 9.864
EcoTrail Paris 2026 (81.4 km) — v_ref=9.86 km/h
/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/Imperial trail de Fontainebleau_2025-09-13 06:01:00.fit
  Nb records : 33857
  Champs dispo : {'distance', 'timestamp'}
  enhanced_altitude sample : [None, 87.0, 86.60000000000002, 86.60000000000002, 86.60000000000002]
  speeds shape: (33846,), dists shape: (33846,)
  total: 70.533
  slopes_all shape: (33857,)
  slopes_interp shape: (33846,)
  v_minetti_ratio shape: (33846,)
  mask_ref sum: 3927
  v_ref: 8.46
Imperial Fontainebleau 2025 (70.5 k

In [3]:
import plotly.graph_objects as go

COLORS = ['#3498db', '#e67e22', '#2ecc71']


def plot_fatigue_profiles(profiles):
    """Plot v_real/v_minetti ratio vs distance percentage for each race."""
    fig = go.Figure()

    for (name, data), color in zip(profiles.items(), COLORS):
        centers = data['centers'] * 100
        ratios  = data['ratios']

        fig.add_trace(go.Scatter(
            x=centers,
            y=ratios,
            mode='lines+markers',
            name=name,
            line=dict(color=color, width=2),
            marker=dict(size=5),
            hovertemplate='%{x:.0f}% : ratio=%{y:.3f}<extra>' + name + '</extra>',
        ))

    # Ligne de référence ratio=1
    fig.add_hline(
        y=1.0,
        line=dict(color='white', width=1, dash='dot'),
        annotation_text='ratio = 1 (pas de fatigue)',
        annotation_position='top right',
    )

    fig.update_layout(
        template='plotly_dark',
        title='Profil de fatigue empirique — ratio v_réelle / v_Minetti',
        xaxis=dict(title='Distance parcourue (%)', range=[0, 100]),
        yaxis=dict(title='Ratio vitesse réelle / Minetti', range=[0.3, 1.3]),
        height=450,
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
    )
    return fig


plot_fatigue_profiles(profiles).show()

In [4]:
from scipy.optimize import curve_fit


def sigmoid_fatigue(x, k, x0):
    """
    Sigmoid fatigue model : f(x) = 1 / (1 + exp(k*(x - x0)))

    x  : fraction de distance [0, 1]
    k  : pente de la dégradation (>0)
    x0 : point d'inflexion (fraction de distance)
    """
    return 1.0 / (1.0 + np.exp(k * (x - x0)))






# Ajustement sur la moyenne des 3 courses
all_centers = profiles[list(profiles.keys())[0]]['centers']
all_ratios  = np.array([p['ratios'] for p in profiles.values()])
mean_ratios = np.nanmean(all_ratios, axis=0)

# Normalisation : le ratio de départ doit être ~1
ref_val = np.nanmedian(mean_ratios[:4])
mean_ratios_norm = mean_ratios / ref_val

valid = ~np.isnan(mean_ratios_norm)
popt, _ = curve_fit(
    sigmoid_fatigue,
    all_centers[valid],
    mean_ratios_norm[valid],
    p0=[5.0, 0.7],
    bounds=([0.5, 0.3], [20.0, 0.95]),
    maxfev=5000,
)
k_fit, x0_fit = popt

print(f"Modèle sigmoïde ajusté :")
print(f"  k  = {k_fit:.3f}  (pente de dégradation)")
print(f"  x0 = {x0_fit:.3f}  (inflexion à {x0_fit*100:.0f}% de la distance)")
print()

# Visualisation de l'ajustement
x_plot = np.linspace(0, 1, 200)
y_fit  = sigmoid_fatigue(x_plot, k_fit, x0_fit)

fig = go.Figure()

for (name, data), color in zip(profiles.items(), COLORS):
    fig.add_trace(go.Scatter(
        x=data['centers'] * 100,
        y=data['ratios'] / np.nanmedian(data['ratios'][:4]),
        mode='markers',
        name=name,
        marker=dict(color=color, size=6, opacity=0.6),
    ))

fig.add_trace(go.Scatter(
    x=x_plot * 100,
    y=y_fit,
    mode='lines',
    name='Modèle sigmoïde',
    line=dict(color='white', width=2.5),
))

fig.add_hline(y=1.0, line=dict(color='gray', width=1, dash='dot'))

fig.update_layout(
    template='plotly_dark',
    title=f'Ajustement sigmoïde — k={k_fit:.2f}, x0={x0_fit:.2f}',
    xaxis=dict(title='Distance parcourue (%)'),
    yaxis=dict(title='Facteur de fatigue normalisé', range=[0.3, 1.3]),
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()

# Paramètres à reporter dans la cellule de paramètres du notebook principal
print(f"\nParamètres à utiliser dans predict_segments() :")
print(f"  FATIGUE_K  = {k_fit:.3f}")
print(f"  FATIGUE_X0 = {x0_fit:.3f}")

Modèle sigmoïde ajusté :
  k  = 3.592  (pente de dégradation)
  x0 = 0.950  (inflexion à 95% de la distance)




Paramètres à utiliser dans predict_segments() :
  FATIGUE_K  = 3.592
  FATIGUE_X0 = 0.950


In [5]:
from scipy.optimize import curve_fit


def plateau_fatigue(x, a, k):
    """
    Exponential fatigue model with plateau.

    f(x) = a + (1 - a) * exp(-k * x)

    x : fraction de distance [0, 1]
    a : facteur asymptotique (niveau de fatigue en fin de course)
    k : vitesse de dégradation initiale
    """
    return a + (1 - a) * np.exp(-k * x)


# Moyenne des 3 courses
all_centers = profiles[list(profiles.keys())[0]]['centers']
all_ratios  = np.array([p['ratios'] for p in profiles.values()])
mean_ratios = np.nanmean(all_ratios, axis=0)

ref_val = np.nanmedian(mean_ratios[:4])
mean_ratios_norm = mean_ratios / ref_val

valid = ~np.isnan(mean_ratios_norm)
popt, _ = curve_fit(
    plateau_fatigue,
    all_centers[valid],
    mean_ratios_norm[valid],
    p0=[0.85, 5.0],
    bounds=([0.5, 0.5], [1.0, 20.0]),
    maxfev=5000,
)
a_fit, k_fit = popt

print(f"Modèle exponentiel à plateau ajusté :")
print(f"  a = {a_fit:.3f}  (plateau asymptotique)")
print(f"  k = {k_fit:.3f}  (vitesse de dégradation)")

# Visualisation
x_plot = np.linspace(0, 1, 200)
y_fit  = plateau_fatigue(x_plot, a_fit, k_fit)

fig = go.Figure()

for (name, data), color in zip(profiles.items(), COLORS):
    fig.add_trace(go.Scatter(
        x=data['centers'] * 100,
        y=data['ratios'] / np.nanmedian(data['ratios'][:4]),
        mode='markers',
        name=name,
        marker=dict(color=color, size=6, opacity=0.6),
    ))

fig.add_trace(go.Scatter(
    x=x_plot * 100,
    y=y_fit,
    mode='lines',
    name='Modèle plateau',
    line=dict(color='white', width=2.5),
))

fig.add_hline(y=1.0, line=dict(color='gray', width=1, dash='dot'))
fig.add_hline(
    y=a_fit,
    line=dict(color='#e74c3c', width=1, dash='dash'),
    annotation_text=f'plateau = {a_fit:.2f}',
    annotation_position='right',
)

fig.update_layout(
    template='plotly_dark',
    title=f'Modèle de fatigue exponentiel — a={a_fit:.3f}, k={k_fit:.3f}',
    xaxis=dict(title='Distance parcourue (%)'),
    yaxis=dict(title='Facteur de fatigue normalisé', range=[0.3, 1.3]),
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)
fig.show()

print(f"\nParamètres à utiliser dans predict_segments() :")
print(f"  FATIGUE_A = {a_fit:.3f}")
print(f"  FATIGUE_K = {k_fit:.3f}")

Modèle exponentiel à plateau ajusté :
  a = 0.753  (plateau asymptotique)
  k = 2.981  (vitesse de dégradation)



Paramètres à utiliser dans predict_segments() :
  FATIGUE_A = 0.753
  FATIGUE_K = 2.981
